In [64]:
import pandas as pd
from datetime import datetime, timedelta
import os

In [65]:
file1_path = r'C:\Users\USER\Downloads\Emarath_global\3CX_Calling_Report_Dataset\call_reports.csv'
file2_path = r'C:\Users\USER\Downloads\Emarath_global\Lead_Data\lead-data.xlsx'
df1 = pd.read_csv(file1_path)
df2 = pd.read_excel(file2_path)

In [66]:
df1.head(2)

,Call Time,Call ID,From,To,Direction,Status,Ringing,Talking,Cost,Call Activity Details,Sentiment,Summary,Transcription
0,2026-03-11T17:44:24,00000000-01dc-b150-99c6-19420000465c,Najiya Emarathglobal (124),00966564492748,Outbound,Unanswered,00:00:03,00:00:00,0.0,Dialed: (00966564492748) → Via rule: Outbound...,NaN,NaN,NaN
1,2026-03-11T17:44:15,00000000-01dc-b150-9403-d5350000465b,Najiya Emarathglobal (124),00966564492748,Outbound,Unanswered,00:00:06,00:00:00,0.0,Dialed: (00966564492748) → Via rule: Outbound...,NaN,NaN,NaN


In [67]:
df2.head(2)

,Name,Phone number,Tags,Agent Phone Number,not contacted,Source,Opt out,Customer Date Created,Customer blocked status,First message received at,Whatsapp Name,First message sent at,Last message sent at,Last CTWA lead at,Agent Name,Last template sent at
0,rasheedahmed 993 pk,9.231184e+11,CTWA,918089898567,NaN,WHATSAPP_CLOUD_API,False,11-02-2026 9:58 pm,False,11-02-2026 9:58 pm,Rasheed Ahmad Kpk,13-02-2026 4:46 pm,11-03-2026 11:59 am,11-03-2026 9:57 am,NaN,NaN
1,موسي ابورماز,9.665971e+11,CTWA,918089898567,NaN,WHATSAPP_CLOUD_API,False,10-03-2026 8:19 pm,False,10-03-2026 8:19 pm,موسي ابورماز,10-03-2026 8:19 pm,11-03-2026 2:11 pm,10-03-2026 8:19 pm,ZAKIYA GAFOOR,NaN


In [68]:


# 1. Helper: Clean Phone Numbers
def normalize_number(phone):
    # Handles strings, floats (scientific notation), and removes non-numeric chars
    phone = str(phone).replace('.0', '')
    phone = ''.join(filter(str.isdigit, phone))
    return phone.lstrip('0')

# Apply normalization
df1['clean_phone'] = df1['To'].apply(normalize_number)
df2['clean_phone'] = df2['Phone number'].apply(normalize_number)

# 2. Categorize Numbers
def categorize_phone(phone):
    # Indian: 10 digits or starts with 91
    if len(phone) == 10 or phone.startswith('91'):
        return 'Indian'
    # Exclude: UAE (971), Bahrain (973), Saudi (966), Qatar (974)
    excluded = ['971', '973', '966', '974']
    if any(phone.startswith(code) for code in excluded):
        return 'Excluded'
    return 'Other'

df2['category'] = df2['clean_phone'].apply(categorize_phone)

# 3. Define Mappings and Filter
name_mapping = {
    'Habiya Emarathglobal (122)': 'HABIYA FARHAN',
    'Shihad Emarathglobal (133)': 'SHIHAD',
    'Rahiyad Emarathglobal (136)': 'RAHIYAD',
    'Safwan K P Emarathglobal (137)': 'MUHAMMED SAFAN K P',
    'Amina Emarathglobal (120)': 'AMINA NISANA V C',
    'Adwaitha Emarathglobal (134)': 'ADWAITHA T M',
    'Najiya Emarathglobal (124)': 'NAJIYA',
    'Arshad Emarathglobal (121)': 'MUHAMMED ARSHAD',
    'Chaithanya Emarathglobal (127)': 'CHAITHANYA P K',
    'Burhana Emarathglobal (119)': 'BURHANA N R',
    'Rahib Emarathglobal (125)': 'MOHAMMED RAHIB K E',
    'Rinsy Emarathglobal (131)': 'RINSY HUSSAIN',
    'Zakiya Emarathglobal (138)': 'ZAKIYA GAFOOR',
    'Akash Emarathglobal (123)': 'AKASH'
}

# Keep only leads assigned to your mapped agents
valid_agents = list(name_mapping.keys())
df1_filtered = df1[df1['From'].isin(valid_agents)].copy()
df1_filtered['standard_agent'] = df1_filtered['From'].map(name_mapping)

# 4. Determine "Called" Status
called_numbers = set(df2['clean_phone'].dropna())
df1_filtered['is_called'] = df1_filtered['clean_phone'].isin(called_numbers)

# 5. Merge to get Category Info
merged_df = pd.merge(df1_filtered, df2[['clean_phone', 'category']], on='clean_phone', how='left')

# 

# 6. Aggregate final report
report = merged_df.groupby('standard_agent').agg(
    not_called_count=('is_called', lambda x: (~x).sum()),
    indian_number_count=('category', lambda x: (x == 'Indian').sum()),
    other_number_count=('category', lambda x: (x == 'Other').sum())
).reset_index()

report

,standard_agent,not_called_count,indian_number_count,other_number_count
0,ADWAITHA T M,53,2,0
1,AKASH,23,0,0
2,AMINA NISANA V C,59,0,11
3,BURHANA N R,51,4,11
4,CHAITHANYA P K,21,6,0
5,HABIYA FARHAN,73,0,285
6,MOHAMMED RAHIB K E,43,0,0
7,MUHAMMED ARSHAD,15,0,0
8,MUHAMMED SAFAN K P,91,0,1
9,NAJIYA,54,1,0


In [69]:
report

,standard_agent,not_called_count,indian_number_count,other_number_count
0,ADWAITHA T M,53,2,0
1,AKASH,23,0,0
2,AMINA NISANA V C,59,0,11
3,BURHANA N R,51,4,11
4,CHAITHANYA P K,21,6,0
5,HABIYA FARHAN,73,0,285
6,MOHAMMED RAHIB K E,43,0,0
7,MUHAMMED ARSHAD,15,0,0
8,MUHAMMED SAFAN K P,91,0,1
9,NAJIYA,54,1,0


In [70]:
# df1[df1['From'].str.lower().str.("emarathglobal", na=False)]['From'].unique()
# Expected: ['Habiya Emarathglobal (122)', 'Burhana Emarathglobal (119)', ...]
# df1['From'].unique()

In [71]:
df2 = df2[['Agent Name','Phone number']].copy()
df2.head(20)

,Agent Name,Phone number
0,NaN,9.231184e+11
1,ZAKIYA GAFOOR,9.665971e+11
2,RINSY HUSSAIN,9.665087e+11
3,SHIHAD,9.665961e+11
4,MUHAMMED ARSHAD,9.715586e+11
5,CHAITHANYA P K,9.665467e+11
6,RAHIYAD,9.665393e+11
7,AMINA NISANA V C,9.665511e+11
8,HABIYA FARHAN,9.665954e+11
9,BURHANA N R,9.665039e+11


In [72]:


# --- CONFIGURATION ---
file1_path = 'first_file.xlsx'
file2_path = 'second_file.xlsx'
output_folder = './output'
phone_column_name = 'Phone Number'  # Ensure this matches your column header exactly

# 1. SET UP TIMEFRAME (Yesterday 3:30 PM onwards)
# Get yesterday's date, then set the time to 15:30 (3:30 PM)
yesterday = datetime.now() - timedelta(days=1)
start_time = yesterday.replace(hour=15, minute=30, second=0, microsecond=0)

print(f"Filtering First File for data from: {start_time}")

# 2. LOAD AND FILTER FIRST FILE
df1 = pd.read_excel(file1_path)

# Ensure the Date column is in datetime format (replace 'Date' with your actual column name)
# If your first file doesn't have a timestamp, you'll need one to filter by 3:30 PM
df1['Date'] = pd.to_datetime(df1['Date']) 
df1_filtered = df1[df1['Date'] >= start_time]

# 3. LOAD SECOND FILE
df2 = pd.read_excel(file2_path)

# 4. COMPARE AND FIND MISSING NUMBERS
# We want numbers in df2 that are NOT in df1_filtered
# First, convert phone columns to strings to ensure matching works (avoids float/int issues)
set1 = set(df1_filtered[phone_column_name].astype(str).str.strip())
set2 = df2[phone_column_name].astype(str).str.strip()

# Filter df2 to keep only rows where the phone number is NOT in set1
missing_leads_df = df2[~set2.isin(set1)]

# 5. EXPORT RESULTS
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

output_path = os.path.join(output_folder, f"Missing_Leads_{datetime.now().strftime('%Y%m%d')}.xlsx")
missing_leads_df.to_excel(output_path, index=False)

print(f"Process Complete!")
print(f"Rows in Second File: {len(df2)}")
print(f"Rows not in First File: {len(missing_leads_df)}")
print(f"Saved to: {output_path}")

Filtering First File for data from: 2026-03-10 15:30:00


FileNotFoundError: [Errno 2] No such file or directory: 'first_file.xlsx'